# 05 homa wire full

Question: does the full wire form of the crate `homa` write and read every packet type as
`homa_wire.h` of HomaModule lays it out at the pinned commit, and refuse every malformed packet
with the error the decoding rules name.

The side under test is `aosabi-devtool homa-packet`, which encodes and decodes one packet through
`homa::wire::Full` per request line, and hands packets to an endpoint of the Network profile with
its operation `take`. The reference side is `tests/harness/homa/full.py`, written from `homa_wire.h`
at HomaModule commit `50d65305aeef9b6ae5cf193a808c7e95f9b1dcb3` and sharing no code with the crate.

This is the synthetic half of stage H1. Four sections run. One packet of each of the ten types goes
through both sides with the header bytes each writes, 1,000 seeded valid packets with their ports go
through both sides in both directions, 64 hostile packets are refused with the error each one is
written to meet or read as FREEZE, which an endpoint drops and counts, and a seeded run of damaged
and random byte strings goes through both decoders, whose answers must agree. The capture half of
stage H4 replays the packets HomaModule sends, which the interop half of `29_homa_raw` captures into
`tests/goldens/homa/interop/`, through both decoders and reads each one back field for field, the
ports among them.

The exit number of the synthetic half, stage H1 of the plan's section The notebooks: 0 byte
differences, 64 of 64 hostile packets refused with the expected error or counted as FREEZE, and
0 panics.

The exit number of the capture half, stage H4: every captured packet read back field for field by
the crate and the reference alike, the ports among them, as many read back as captured.

Judged stages: H1 and H4 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The seed of the valid packets and of the damaged byte strings.
seed = 20260927

# The valid packets of the exit number.
max_ops = 1000

# The damaged and random byte strings both decoders read.
fuzz_ops = 10000

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the installed targets and the gates of the plan's section The
commands that the notebook rests on: the build of the devtool, the lints of the workspace and the
format of the crates `homa` and `aosabi-devtool`, every cargo call with `--locked`. Every batch is
written under `tests/notebooks/out/05_homa_wire_full/`, which git ignores, so a failing case is
replayed by hand with the command the notebook runs.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import bytediff, paths, run
from harness.homa import full

P = paths(aosabi_root)
OUT = P.out / "05_homa_wire_full"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, and the format of the crate homa under its own edition 2021
# and of the devtool, which rustfmt checks because cargo fmt takes no
# --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-packet", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("batches        :", P.rel(OUT))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, "the pinned toolchain is %s, this is %s" % (
    rust_version,
    reported_cargo,
)
assert smoke == {"subcommand": "homa-packet", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []
DIFFS = {}


def batch(requests, name):
    """Send `requests` to `homa-packet` in one process and return its answers.

    A process that exits other than 0 or answers fewer lines than it read
    is a panic or a crash of the seam, which the verdict counts.
    """
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-packet", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    fatal = [a for a in answers if "fatal" in a]
    assert not fatal, "the seam refused requests of %s: %s" % (name, fatal[:3])
    return answers


def python_encode(packet, ports):
    encoded = full.encode(packet, ports)
    return "NoRoom" if encoded is None else encoded.hex()


def python_decode(hex_bytes):
    decoded, error = full.decode(bytes.fromhex(hex_bytes))
    return {"error": error} if error else {"decoded": decoded}


def as_double(value):
    """Return `value` with every integer of 2^53 or more read as the double
    the batch form carries it as, since a JSON number is a double."""
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        return float(value) if abs(value) >= 2 ** 53 else value
    if isinstance(value, dict):
        return {k: as_double(v) for k, v in value.items()}
    if isinstance(value, list):
        return [as_double(v) for v in value]
    return value


def both_ways(name, packets):
    """Encode `packets`, each with its ports, on both sides, decode each
    side's bytes on the other, and compare the streams byte for byte."""
    requests = [
        {"id": i, "op": "encode", "form": "full", "packet": p, "ports": ports}
        for i, (p, ports) in enumerate(packets)
    ]
    encoded = batch(requests, name + "-encode")
    rust = [a.get("hex", a.get("error")) for a in encoded]
    python = [python_encode(p, ports) for p, ports in packets]
    differing = [i for i, (r, q) in enumerate(zip(rust, python)) if r != q]
    readable = [i for i, h in enumerate(python) if h != "NoRoom"]
    decode_rust = batch(
        [{"id": i, "op": "decode", "form": "full", "hex": python[i]} for i in readable],
        name + "-decode",
    )
    wrong_rust = [
        i for i, a in zip(readable, decode_rust)
        if a.get("decoded") != {"ports": packets[i][1], "packet": packets[i][0], "need_limits": False}
    ]
    wrong_python = [
        i for i in readable
        if rust[i] == "NoRoom" or python_decode(rust[i]) != {
            "decoded": {"ports": packets[i][1], "packet": packets[i][0], "need_limits": False}}
    ]
    DIFFS[name] = len(differing) + len(wrong_rust) + len(wrong_python)
    print("%s: %d packets, %d encodings differ, %d packets the crate reads back wrong,"
          " %d the reference reads back wrong" % (
              name, len(packets), len(differing), len(wrong_rust), len(wrong_python)))
    for i in (differing + wrong_rust + wrong_python)[:3]:
        print("  case", i, json.dumps(packets[i][0])[:160])
        print("    crate    ", rust[i][:120])
        print("    reference", python[i][:120])
    bytediff(
        "\n".join(python).encode(), "\n".join(rust).encode(), name + " encodings, reference against crate"
    )
    return rust

## The ten packet types

One packet of each type of `homa_wire.h`, DATA as a whole message, as a part with an acknowledgement
and with the incast mark, GRANT, RESEND, `RPC_UNKNOWN`, BUSY, CUTOFFS, FREEZE, `NEED_ACK`, ACK with
no entry and with five, and `START_MSG`, and LIMITS, which the compact form has and this one does
not. Neither side writes FREEZE, which the crate never sends, nor LIMITS, and both answer `NoRoom`
for them. The header bytes each type writes are printed beside and held to the sizes of the packed
structures of `homa_wire.h`: 56 for DATA, 33 for GRANT, 37 for RESEND, 28 for the four packets of
the common header alone, 62 for CUTOFFS, 80 for ACK whatever its entries and 32 for `START_MSG`.

In [ ]:
JAPANESE = [
    "日本語のテキスト",
    "東京都渋谷区神南一丁目",
    "こんにちは、世界。",
    "漢字とひらがなとカタカナ",
    "{\"data\":{\"名前\":\"山田太郎\"}}",
]


def data_packet(ident, payload, msg_length=None, offset=0, ack=None, retransmit=False,
                cutoff_version=0, incast=False):
    return {
        "type": "DATA",
        "id": ident,
        "msg_length": len(payload) if msg_length is None else msg_length,
        "offset": offset,
        "retransmit": retransmit,
        "ack": ack,
        "cutoff_version": cutoff_version,
        "incast": incast,
        "payload": payload.hex(),
    }


PORTS = {"src": 32768, "dst": 3000}
TYPES = [
    ("DATA whole", data_packet(0x1000_0002, JAPANESE[0].encode(), cutoff_version=1)),
    ("DATA part, ACK, incast", data_packet(
        0x1000_0003, JAPANESE[1].encode(), msg_length=40_000, offset=1_424,
        ack={"client_id": 0x1000_0000, "server_port": 3000}, retransmit=True, incast=True)),
    ("GRANT", {"type": "GRANT", "id": 0x1000_0003, "offset": 50_000, "priority": 3}),
    ("RESEND", {"type": "RESEND", "id": 0x1000_0002, "offset": 0, "length": 0xFFFF_FFFF, "priority": 7}),
    ("RPC_UNKNOWN", {"type": "RPC_UNKNOWN", "id": 0x1000_0003}),
    ("BUSY", {"type": "BUSY", "id": 0x1000_0003}),
    ("CUTOFFS", {"type": "CUTOFFS", "cutoffs": [0, 0, 0, 0, 1_000_000, 15_000, 2_800, 200], "version": 1}),
    ("NEED_ACK", {"type": "NEED_ACK", "id": 0x1000_0003}),
    ("ACK of 0", {"type": "ACK", "id": 0x1000_0002, "acks": []}),
    ("ACK of 5", {"type": "ACK", "id": 0x1000_0002,
                  "acks": [{"client_id": 2 * k + 4, "server_port": 3000 + k} for k in range(5)]}),
    ("START_MSG", {"type": "START_MSG", "id": 0x1000_0004, "msg_length": 250_000}),
]
UNWRITTEN = [
    ("FREEZE", {"type": "FREEZE"}),
    ("LIMITS", {"type": "LIMITS", "max_packet": 1500, "max_message": 1_000_000,
                "unsched_bytes": 40_000, "max_rpcs": 8}),
]
type_rust = both_ways("types", [(p, PORTS) for _, p in TYPES])
unwritten = batch(
    [{"id": i, "op": "encode", "form": "full", "packet": p, "ports": PORTS} for i, (_, p) in enumerate(UNWRITTEN)],
    "types-unwritten",
)
print()
print("%-24s %-12s %-12s %s" % ("packet", "header bytes", "homa_wire.h", "crate bytes"))
HEADERS = {"DATA": 56, "GRANT": 33, "RESEND": 37, "RPC_UNKNOWN": 28, "BUSY": 28, "CUTOFFS": 62,
           "NEED_ACK": 28, "ACK": 80, "START_MSG": 32}
header_differences = 0
for (label, packet), hexed in zip(TYPES, type_rust):
    payload = len(bytes.fromhex(packet.get("payload", ""))) if packet["type"] == "DATA" else 0
    written = len(bytes.fromhex(hexed)) - payload
    header_differences += written != HEADERS[packet["type"]] or written != full.LENGTHS[full.CODES[packet["type"]]]
    print("%-24s %-12d %-12d %s" % (label, written, HEADERS[packet["type"]], hexed[:56]))
for (label, packet), answer in zip(UNWRITTEN, unwritten):
    reference = python_encode(packet, PORTS)
    header_differences += answer.get("error") != "NoRoom" or reference != "NoRoom"
    print("%-24s %-12s %-12s crate %s, reference %s" % (label, "-", "-", answer.get("error", answer.get("hex")), reference))
DIFFS["headers"] = header_differences
kinds = sorted({p["type"] for _, p in TYPES + UNWRITTEN})
print()
print("types covered:", len(kinds), kinds)
assert len(kinds) == 11, kinds

## 1,000 seeded valid packets

The generator draws the type, the ports, the identifier below 2^53, which the batch form carries
exactly, and every field inside its range: a level of 0 to 7, a cutoff version of 16 bits, an
acknowledgement or none, an ACK of 0 to 5 entries and the eight cutoffs of CUTOFFS. A DATA packet
carries seeded bytes or a run of Japanese text, up to one packet of 1,424 bytes, whole or as a
part whose offset and payload stay inside its length, and a message that is empty carries no
payload. Each packet goes through both sides in both directions, and the streams of both encoders
are compared byte for byte.

In [ ]:
def random_payload(g):
    if g.random() < 0.3:
        text = g.choice(JAPANESE).encode() * g.randint(1, 30)
        return text[: g.randint(1, min(len(text), 1424))]
    return bytes(g.getrandbits(8) for _ in range(g.randint(1, g.choice([40, 300, 1424]))))


def random_ack(g):
    if g.random() < 0.5:
        return None
    return {"client_id": g.randint(1, 2 ** 53 - 1), "server_port": g.getrandbits(16)}


def random_packet(g):
    kind = g.choice(["DATA", "DATA", "DATA", "GRANT", "RESEND", "RPC_UNKNOWN", "BUSY", "CUTOFFS",
                     "NEED_ACK", "ACK", "START_MSG"])
    ident = g.randint(0, 2 ** 53 - 1)
    if kind == "DATA":
        extra = dict(ack=random_ack(g), retransmit=g.random() < 0.2,
                     cutoff_version=g.getrandbits(16), incast=g.random() < 0.2)
        if g.random() < 0.05:
            return data_packet(ident, b"", **extra)
        payload = random_payload(g)
        if g.random() < 0.4:
            return data_packet(ident, payload, **extra)
        msg_length = g.randint(len(payload) + 1, min(len(payload) + 1_000_000, 2 ** 32 - 1))
        offset = g.randint(0, msg_length - len(payload))
        return data_packet(ident, payload, msg_length=msg_length, offset=offset, **extra)
    if kind == "GRANT":
        return {"type": "GRANT", "id": ident, "offset": g.getrandbits(32), "priority": g.randint(0, 7)}
    if kind == "RESEND":
        return {"type": "RESEND", "id": ident, "offset": g.getrandbits(32), "length": g.getrandbits(32),
                "priority": g.randint(0, 7)}
    if kind == "CUTOFFS":
        return {"type": "CUTOFFS", "cutoffs": [g.getrandbits(32) for _ in range(8)], "version": g.getrandbits(16)}
    if kind == "ACK":
        return {"type": "ACK", "id": ident, "acks": [
            {"client_id": g.randint(0, 2 ** 53 - 1), "server_port": g.getrandbits(16)}
            for _ in range(g.randint(0, 5))]}
    if kind == "START_MSG":
        return {"type": "START_MSG", "id": ident, "msg_length": g.getrandbits(32)}
    return {"type": kind, "id": ident}


generator = random.Random(seed)
valid = [(random_packet(generator), {"src": generator.getrandbits(16), "dst": generator.getrandbits(16)})
         for _ in range(max_ops)]
by_type = {}
for packet, _ in valid:
    by_type[packet["type"]] = by_type.get(packet["type"], 0) + 1
japanese = sum(
    1 for p, _ in valid
    if p["type"] == "DATA" and any(t.encode()[:6] in bytes.fromhex(p["payload"]) for t in JAPANESE)
)
print("valid packets:", len(valid), "by type:", by_type)
print("DATA packets that carry Japanese text:", japanese)
valid_rust = both_ways("valid", valid)

## 64 hostile packets

Each hostile packet is written to break one rule of the decoder, and the table names the error the
rules give it: Short below the 28 bytes of the common header, Form for LIMITS and Type for any
other code outside 0x10 to 0x19, Reserved for the marker of TCP hijacking in `urgent` and for a
pad bit of DATA other than the incast mark, Length for a packet shorter than its type's header,
a control packet longer than its header and DATA of a message that is not empty with no payload,
and Range for a level above 7, an ACK of more than five entries and a DATA payload past its
length. The list holds every type short by one and every control type long by one, the types
`0x0f` and `0x1a`, an ACK with `num_acks` 6, a GRANT at level 8, DATA whose offset runs past its
length, the hijack marker in `urgent`, and six packets that break two rules to fix the order in
which the rules are named. The crate's answer and the reference's answer must both equal the
expected name.

Two of the 64 are FREEZE, which both decoders read, since `homa_wire.h` has the type, and which an
endpoint drops as `Refused::Freeze` and counts in `Stats::freezes_dropped`. The 64 then go to an
endpoint of the Network profile through `take`, which must refuse every one, the 62 malformed ones
as `Malformed` with the same error and the two FREEZE as `Freeze`, each counted.

In [ ]:
def control(kind, ident=3, length=None, urgent=0, src=40000, dst=3000):
    """A packet of the common header with the fields a control packet uses
    zero, `length` bytes long, its type's header by default."""
    size = full.LENGTHS.get(kind, 28) if length is None else length
    body = bytearray(max(size, 28))
    body[0:2] = src.to_bytes(2, "big")
    body[2:4] = dst.to_bytes(2, "big")
    body[11] = kind
    body[12] = full.DOFF_DATA if kind == full.DATA else full.DOFF_CONTROL
    body[18:20] = urgent.to_bytes(2, "big")
    body[20:28] = ident.to_bytes(8, "big")
    return bytes(body[:size])


def data(msg_length=10, offset=0, payload=b"0123456789", pad=(0, 0, 0), urgent=0):
    head = bytearray(control(full.DATA, urgent=urgent))
    head[28:32] = msg_length.to_bytes(4, "big")
    head[49:52] = bytes(pad)
    head[52:56] = offset.to_bytes(4, "big")
    return bytes(head) + payload


def with_byte(packet, at, value):
    body = bytearray(packet)
    body[at] = value
    return bytes(body)


grant = control(full.GRANT)
ack = with_byte(control(full.ACK), 29, 5)
HOSTILE = []
for n in (0, 1, 11, 20, 27):
    HOSTILE.append(("truncated to %d bytes" % n, grant[:n], "Short"))
for kind in (full.GRANT, full.RESEND, full.CUTOFFS, full.ACK, full.START_MSG):
    HOSTILE.append(("%s short by one" % full.TYPES[kind], control(kind)[:-1], "Length"))
for kind in (full.RPC_UNKNOWN, full.BUSY, full.FREEZE, full.NEED_ACK):
    HOSTILE.append(("%s short by one" % full.TYPES[kind], control(kind)[:-1], "Short"))
HOSTILE.append(("DATA header short by one, no payload", data(msg_length=0, payload=b"")[:-1], "Length"))
HOSTILE.append(("DATA of the common header alone", control(full.DATA, length=28), "Length"))
HOSTILE.append(("DATA cut before its pad", data()[:49], "Length"))
HOSTILE.append(("DATA cut inside its pad", data()[:51], "Length"))
HOSTILE.append(("DATA cut after its pad", data()[:52], "Length"))
for kind in (full.GRANT, full.RESEND, full.RPC_UNKNOWN, full.BUSY, full.CUTOFFS, full.FREEZE,
             full.NEED_ACK, full.ACK, full.START_MSG):
    HOSTILE.append(("%s long by one" % full.TYPES[kind], control(kind) + b"\x00", "Length"))
HOSTILE.append(("CUTOFFS long by eight", control(full.CUTOFFS) + bytes(8), "Length"))
HOSTILE.append(("CUTOFFS short by eight", control(full.CUTOFFS)[:-8], "Length"))
for kind in (0x00, 0x0F, 0x1A, 0x21, 0xFF):
    HOSTILE.append(("type 0x%02x" % kind, control(kind, length=28), "Type"))
HOSTILE.append(("LIMITS, of the compact form", control(full.LIMITS, length=28), "Form"))
HOSTILE.append(("LIMITS of 20 bytes, short first", control(full.LIMITS, length=28)[:20], "Short"))
HOSTILE.append(("ACK with num_acks 6", with_byte(ack, 29, 6), "Range"))
HOSTILE.append(("ACK with num_acks 65535", with_byte(with_byte(ack, 28, 0xFF), 29, 0xFF), "Range"))
HOSTILE.append(("GRANT at level 8", with_byte(grant, 32, 8), "Range"))
HOSTILE.append(("GRANT at level 255", with_byte(grant, 32, 255), "Range"))
HOSTILE.append(("RESEND at level 8", with_byte(control(full.RESEND), 36, 8), "Range"))
HOSTILE.append(("RESEND at level 200", with_byte(control(full.RESEND), 36, 200), "Range"))
HOSTILE.append(("DATA whose offset runs past its length", data(msg_length=10, offset=5), "Range"))
HOSTILE.append(("DATA whose end passes 2^32", data(msg_length=0xFFFF_FFFF, offset=0xFFFF_FFFE), "Range"))
HOSTILE.append(("DATA of msg_length 0 with a payload", data(msg_length=0), "Range"))
HOSTILE.append(("DATA of an empty message at offset 5", data(msg_length=0, offset=5, payload=b""), "Range"))
HOSTILE.append(("DATA of 10 bytes with no payload", data(msg_length=10, payload=b""), "Length"))
HOSTILE.append(("DATA with pad bit 1", data(pad=(2, 0, 0)), "Reserved"))
HOSTILE.append(("DATA with every pad bit but the mark", data(pad=(0xFE, 0, 0)), "Reserved"))
HOSTILE.append(("DATA with its second pad byte set", data(pad=(1, 1, 0)), "Reserved"))
HOSTILE.append(("DATA with its third pad byte set", data(pad=(0, 0, 0x80)), "Reserved"))
HOSTILE.append(("GRANT with the hijack marker", control(full.GRANT, urgent=0xB97D), "Reserved"))
HOSTILE.append(("DATA with the hijack marker", data(urgent=0xB97D), "Reserved"))
HOSTILE.append(("FREEZE with the hijack marker", control(full.FREEZE, urgent=0xB97D), "Reserved"))
HOSTILE.append(("type 0x1a with the marker, type first", control(0x1A, length=28, urgent=0xB97D), "Type"))
HOSTILE.append(("27 bytes with the marker, short first", control(full.GRANT, urgent=0xB97D)[:27], "Short"))
HOSTILE.append(("GRANT long by one with the marker, marker first",
                control(full.GRANT, urgent=0xB97D) + b"\x00", "Reserved"))
HOSTILE.append(("LIMITS with the marker, form first", control(full.LIMITS, length=28, urgent=0xB97D), "Form"))
HOSTILE.append(("DATA with a pad bit past its length, pad first",
                data(msg_length=10, offset=5, pad=(4, 0, 0)), "Reserved"))
HOSTILE.append(("GRANT at level 8 long by one, length first", with_byte(grant, 32, 8) + b"\x00", "Length"))
HOSTILE.append(("ACK with num_acks 6 long by one, length first", with_byte(ack, 29, 6) + b"\x00", "Length"))
HOSTILE.append(("FREEZE", control(full.FREEZE, ident=0), "FREEZE"))
HOSTILE.append(("FREEZE that names a sender and ports", control(full.FREEZE, ident=99, src=1, dst=2), "FREEZE"))

print("hostile packets:", len(HOSTILE))
assert len(HOSTILE) == 64, len(HOSTILE)
answers = batch(
    [{"id": i, "op": "decode", "form": "full", "hex": h.hex()} for i, (_, h, _) in enumerate(HOSTILE)],
    "hostile",
)
taken = batch([{"id": 0, "op": "take", "form": "full", "port": 3000,
                "hexes": [h.hex() for _, h, _ in HOSTILE]}], "hostile-take")[0]


def read_as(answer):
    if "error" in answer:
        return answer["error"]
    return answer["decoded"]["packet"]["type"]


def reference_read(data):
    decoded, error = full.decode(data)
    return error if error else decoded["packet"]["type"]


refused = 0
print("%-50s %-9s %-9s %-9s %s" % ("packet", "expected", "crate", "reference", "endpoint"))
for (label, data, expected), answer, outcome in zip(HOSTILE, answers, taken["results"]):
    crate = read_as(answer)
    reference = reference_read(data)
    at_endpoint = "Freeze" if expected == "FREEZE" else "Malformed:" + expected
    ok = crate == expected == reference and outcome == at_endpoint
    refused += ok
    print("%-50s %-9s %-9s %-9s %s%s" % (label[:50], expected, crate, reference, outcome, "" if ok else "  DIFFERS"))
freezes = sum(1 for _, _, expected in HOSTILE if expected == "FREEZE")
print()
print("hostile packets refused with the expected error or read as FREEZE: %d of %d" % (refused, len(HOSTILE)))
print("counters of the endpoint:", taken["stats"])

## Damaged and random byte strings

The seeded run takes the encodings of the valid packets and damages each one, a byte flipped, cut,
dropped, duplicated or appended, and adds strings of random bytes of 0 to 100 bytes, a third of
which carry a type of the form in byte 11 so that they reach the rules behind the type. Both
decoders read every string, and for each one they must both decode the same packet or both name the
same error. A string carries identifiers of all 64 bits, and the batch form carries a number as a
double, so an identifier of 2^53 or more is compared on both sides as the double nearest it, while
the valid packets above draw every identifier below 2^53, where the comparison is exact. The seam
must answer every line, which is the count of panics the exit number asks for.

In [ ]:
fuzz = random.Random(seed ^ 0x05)
strings = []
for n in range(fuzz_ops):
    if n % 4 == 0:
        body = bytearray(fuzz.getrandbits(8) for _ in range(fuzz.randint(0, 100)))
        if len(body) > 11 and fuzz.random() < 1 / 3:
            body[11] = fuzz.randint(0x10, 0x19)
        strings.append(bytes(body))
        continue
    base = bytearray(bytes.fromhex(fuzz.choice([v for v in valid_rust if v != "NoRoom"])))
    action = fuzz.randrange(5)
    if action == 0 and base:
        base[fuzz.randrange(len(base))] ^= 1 << fuzz.randrange(8)
    elif action == 1:
        del base[fuzz.randint(0, len(base)):]
    elif action == 2 and base:
        del base[fuzz.randrange(len(base))]
    elif action == 3 and base:
        at = fuzz.randrange(len(base))
        base[at:at] = base[at:at + 1]
    else:
        base += bytes(fuzz.getrandbits(8) for _ in range(fuzz.randint(1, 8)))
    strings.append(bytes(base))
answers = batch(
    [{"id": i, "op": "decode", "form": "full", "hex": s.hex()} for i, s in enumerate(strings)],
    "fuzz",
)
disagree = []
decoded = 0
for s, answer in zip(strings, answers):
    crate = as_double({k: v for k, v in answer.items() if k in ("decoded", "error")})
    reference = as_double(python_decode(s.hex()))
    decoded += "decoded" in crate
    if crate != reference:
        disagree.append((s.hex(), crate, reference))
DIFFS["fuzz"] = len(disagree)
print("byte strings:", len(strings), " decoded by both:", decoded,
      " refused by both:", len(strings) - decoded - len(disagree))
print("answers that differ:", len(disagree))
for s, crate, reference in disagree[:5]:
    print("  ", s[:80], crate, reference)

## The capture half

The capture half replays the packets HomaModule sent in the interop half of `29_homa_raw` from
`tests/goldens/homa/interop/`, and reads each one back field for field, the ports among them. Every
captured packet goes through the side under test, `aosabi-devtool homa-packet` with its operation
`decode` and the form `full`, and through the reference `full.py`, and the two answers are compared
as the synthetic half compares them. A packet counts as read back where both sides decode it and
read every field alike, and the number read back must equal the number captured. The reference
writes each packet it read again, and the cell prints how many of those headers equal the captured
bytes over the length of the packet's type, which byte 11 names. That count is printed and not
judged, since the full form leaves fields HomaModule may fill unread, the sequence among them, and
the interop half of `29_homa_raw` judges the bytes of the headers.

The cell replays the goldens where they exist and prints not run until then, which is never a pass.

In [ ]:
from harness.homa import pcap


def replay(paths):
    """Read every Homa packet of the captures `paths` on both sides and return
    the packets captured, those both sides read alike field for field, those
    whose header the reference writes again byte for byte over the length of
    their type, and the packets that differ."""
    captured = []
    for path in paths:
        _linktype, found = pcap.read(path)
        captured.extend((path, packet) for packet in found)
    answers = batch([{"id": i, "op": "decode", "form": "full", "hex": packet.homa.hex()}
                     for i, (_, packet) in enumerate(captured)], "capture") if captured else []
    alike, rewritten, differing = 0, 0, []
    for (path, packet), answer in zip(captured, answers):
        crate = as_double({k: v for k, v in answer.items() if k in ("decoded", "error")})
        reference = as_double(python_decode(packet.homa.hex()))
        if "decoded" not in reference or crate != reference:
            differing.append((P.rel(path), packet.homa[:64].hex(), crate, reference))
            continue
        alike += 1
        decoded, _ = full.decode(packet.homa)
        again = full.encode(decoded["packet"], decoded["ports"])
        length = full.LENGTHS.get(packet.homa[11], full.COMMON_LEN)
        rewritten += again is not None and again[:length] == packet.homa[:length]
    return len(captured), alike, rewritten, differing


interop_dir = P.root / "tests" / "goldens" / "homa" / "interop"
captures = sorted(interop_dir.rglob("*.pcap")) if interop_dir.is_dir() else []
read_back = None
if not captures:
    print("not run: the capture half, which replays the packets HomaModule sent in the interop half"
          " of 29_homa_raw, held at not run by owner question 21, HomaModule's tools on aarch64")
else:
    captured, read_back, rewritten, differing = replay(captures)
    print("captures: %d files, %d Homa packets" % (len(captures), captured))
    print("captured packets read back field for field by the crate and the reference, the ports among them:",
          read_back)
    print("headers the reference writes again equal to the captured bytes over the length of their type:"
          " %d of %d, printed" % (rewritten, read_back))
    for path, head, crate, reference in differing[:5]:
        print("  differs:", path, head, "crate", json.dumps(crate)[:160], "reference", json.dumps(reference)[:160])
    DIFFS["capture"] = len(differing)
    assert captured > 0, "the captures hold no Homa packet"
    assert read_back == captured, "%d of %d captured packets read back" % (read_back, captured)

## Verdict

In [ ]:
print("differences by section:", DIFFS)
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
print("hostile packets refused with the expected error or read as FREEZE: %d of %d" % (refused, len(HOSTILE)))
print("FREEZE packets the endpoint dropped and counted: %d of %d" % (taken["stats"]["freezes_dropped"], freezes))
assert not PANICS, PANICS
assert sum(DIFFS.values()) == 0, DIFFS
assert refused == len(HOSTILE) == 64
assert taken["stats"]["freezes_dropped"] == freezes == 2, taken["stats"]
assert taken["stats"]["refused"] == len(HOSTILE), taken["stats"]
print()
capture = ("the capture half not run, owner question 21" if read_back is None
           else "%d captured packets read back field for field" % read_back)
print("VERDICT: 0 byte differences over the packets of the 11 types and %d seeded valid packets, %d of %d hostile"
      " packets refused with the expected error or counted as FREEZE, %d damaged strings read alike,"
      " 0 panics, and %s" % (len(valid), refused, len(HOSTILE), len(strings), capture))